# Olist AI Database Analyst Agent — Codebase Walkthrough

A step-by-step notebook that explores every component of this project.

**What is this project?**
An AI-powered agent that answers natural-language questions against the Olist Brazilian e-commerce SQLite database (`data/olist.db`).
It uses **RAG** (retrieval-augmented generation) to pick the right schema context, generates SQL with an LLM, validates/repairs it, executes it, then summarizes results in natural language.

**Repo layout**
```
olist-ai-agent/
  app/                  # FastAPI & Streamlit front-ends
  src/
    agent/              # Orchestrator, LLM client, SQL gen/validate/repair, analyst
    schema/             # Schema extraction, profiling, relationship discovery, semantic mapping
    retrieval/          # FAISS-backed schema retriever
    analytics/          # Visualization & metrics layer
    evaluation/         # Benchmark datasets, metrics, runner, experiments, robustness
    database/           # SQLAlchemy connection helper
    config.py           # LLM config (env vars)
  scripts/              # One-shot utilities (extract, profile, discover relationships)
  data/                 # olist.db + generated JSON artifacts
  tests/                # Minimal tests
```

We will cover each layer in order.

---
## 0. Setup

In [1]:
import os, sys, json, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO)
logging.getLogger('sqlalchemy.engine').setLevel(logging.WARNING)

# Ensure project root is on sys.path so src.* imports work
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')

Working directory: /Users/dev/olist-ai-agent


---
## 1. Database layer  (`src/database/connection.py`)

Thin wrapper around SQLAlchemy that creates an engine and verifies the connection with `SELECT 1`.

In [2]:
from src.database.connection import create_database_engine, verify_connection

engine = create_database_engine('sqlite:///data/olist.db')
print(f'Engine dialect: {engine.dialect.name}')
print(f'Connection OK:  {verify_connection(engine)}')

INFO:src.database.connection:Creating database engine for URL: sqlite:///data/olist.db
INFO:src.database.connection:Database connection test successful.


Engine dialect: sqlite
Connection OK:  True


List all tables in the Olist database:

In [3]:
from sqlalchemy import text

with engine.connect() as conn:
    tables = [r[0] for r in conn.execute(text(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ))]
print(tables)

['customers', 'geolocation', 'leads_closed', 'leads_qualified', 'order_items', 'order_payments', 'order_reviews', 'orders', 'product_category_name_translation', 'products', 'sellers']


---
## 2. Schema models  (`src/schema/models.py`)

Pure-Python dataclasses that hold extracted metadata — no database dependency.

In [4]:
from src.schema.models import ColumnMetadata, ForeignKeyMetadata, TableMetadata, DatabaseSchema

# Quick look at the model fields
print('ColumnMetadata fields:', [f.name for f in ColumnMetadata.__dataclass_fields__.values()])
print('TableMetadata fields: ', [f.name for f in TableMetadata.__dataclass_fields__.values()])

ColumnMetadata fields: ['name', 'data_type', 'nullable', 'primary_key', 'default', 'comment']
TableMetadata fields:  ['name', 'columns', 'primary_keys', 'foreign_keys', 'indexes']


---
## 3. Schema extraction  (`src/schema/extractor.py`)

`SchemaExtractor` uses SQLAlchemy's `inspect()` to pull columns, PKs, FKs, and indexes for every table.

You can run `scripts/extract_schema.py` to produce `data/schema.json`, or do it inline here.

In [5]:
from src.schema.extractor import SchemaExtractor

extractor = SchemaExtractor(engine)
schema = extractor.extract()

print(f'Database type: {schema.database_type}')
print(f'Tables found:  {len(schema.tables)}')
for t in schema.tables:
    print(f'  {t.name}: {len(t.columns)} cols, {len(t.primary_keys)} PKs, {len(t.foreign_keys)} FKs')

INFO:src.schema.extractor:Starting schema extraction...
INFO:src.schema.extractor:Connected to database type: sqlite
INFO:src.schema.extractor:Found 11 tables: ['customers', 'geolocation', 'leads_closed', 'leads_qualified', 'order_items', 'order_payments', 'order_reviews', 'orders', 'product_category_name_translation', 'products', 'sellers']
INFO:src.schema.extractor:Extracting metadata for table: customers
INFO:src.schema.extractor:Extracting metadata for table: geolocation
INFO:src.schema.extractor:Extracting metadata for table: leads_closed
INFO:src.schema.extractor:Extracting metadata for table: leads_qualified
INFO:src.schema.extractor:Extracting metadata for table: order_items
INFO:src.schema.extractor:Extracting metadata for table: order_payments
INFO:src.schema.extractor:Extracting metadata for table: order_reviews
INFO:src.schema.extractor:Extracting metadata for table: orders
INFO:src.schema.extractor:Extracting metadata for table: product_category_name_translation
INFO:src.s

Database type: sqlite
Tables found:  11
  customers: 5 cols, 0 PKs, 0 FKs
  geolocation: 5 cols, 0 PKs, 0 FKs
  leads_closed: 14 cols, 0 PKs, 0 FKs
  leads_qualified: 4 cols, 0 PKs, 0 FKs
  order_items: 7 cols, 0 PKs, 0 FKs
  order_payments: 5 cols, 0 PKs, 0 FKs
  order_reviews: 7 cols, 0 PKs, 0 FKs
  orders: 8 cols, 0 PKs, 0 FKs
  product_category_name_translation: 2 cols, 0 PKs, 0 FKs
  products: 9 cols, 0 PKs, 0 FKs
  sellers: 4 cols, 0 PKs, 0 FKs


Inspect a single table in detail:

In [6]:
orders_table = next(t for t in schema.tables if t.name == 'orders')
print(f'Table: {orders_table.name}')
print(f'Primary keys: {orders_table.primary_keys}')
print('Columns:')
for col in orders_table.columns:
    print(f'  {col.name:40s}  type={col.data_type:10s}  nullable={col.nullable}  PK={col.primary_key}')
print('Foreign keys:')
for fk in orders_table.foreign_keys:
    print(f'  {fk.constrained_column} -> {fk.referred_table}.{fk.referred_column}')

Table: orders
Primary keys: []
Columns:
  order_id                                  type=TEXT        nullable=True  PK=0
  customer_id                               type=TEXT        nullable=True  PK=0
  order_status                              type=TEXT        nullable=True  PK=0
  order_purchase_timestamp                  type=TEXT        nullable=True  PK=0
  order_approved_at                         type=TEXT        nullable=True  PK=0
  order_delivered_carrier_date              type=TEXT        nullable=True  PK=0
  order_delivered_customer_date             type=TEXT        nullable=True  PK=0
  order_estimated_delivery_date             type=TEXT        nullable=True  PK=0
Foreign keys:


---
## 4. Database profiling  (`src/schema/profiler.py`)

`DatabaseProfiler` samples each table and computes per-column statistics:
- null counts/percentages
- unique counts
- type categorization (`numeric`, `boolean`, `date/time`, `identifier`, `categorical`, `text`)
- min/max/mean/median for numerics
- top value frequencies for categoricals

Run `scripts/profile_database.py` to produce `data/database_profile.json`, or inline:

In [7]:
from src.schema.profiler import DatabaseProfiler

profiler = DatabaseProfiler(engine, schema, sample_size=1000)
profile = profiler.profile_all()

print(f'Tables profiled: {len(profile["tables"])}')
# Show a snippet of the first table profile
print(json.dumps(profile['tables'][0], indent=2, default=str)[:1200])

INFO:src.schema.profiler:Profiling table: customers
INFO:src.schema.profiler:  Profiling column: customer_id
INFO:src.schema.profiler:  Profiling column: customer_unique_id
INFO:src.schema.profiler:  Profiling column: customer_zip_code_prefix
INFO:src.schema.profiler:  Profiling column: customer_city
INFO:src.schema.profiler:  Profiling column: customer_state
INFO:src.schema.profiler:Profiling table: geolocation
INFO:src.schema.profiler:  Profiling column: geolocation_zip_code_prefix
INFO:src.schema.profiler:  Profiling column: geolocation_lat
INFO:src.schema.profiler:  Profiling column: geolocation_lng
INFO:src.schema.profiler:  Profiling column: geolocation_city
INFO:src.schema.profiler:  Profiling column: geolocation_state
INFO:src.schema.profiler:Profiling table: leads_closed
INFO:src.schema.profiler:  Profiling column: mql_id
INFO:src.schema.profiler:  Profiling column: seller_id
INFO:src.schema.profiler:  Profiling column: sdr_id
INFO:src.schema.profiler:  Profiling column: sr_id

Tables profiled: 11
{
  "table": "customers",
  "row_count": 99441,
  "column_count": 5,
  "columns": {
    "customer_id": {
      "type": "identifier",
      "null_count": 0,
      "null_percentage": 0.0,
      "unique_count": 99441,
      "sample_values": [
        "06b8999e2fba1a1fbc88172c00ba8bc7",
        "18955e83d337fd6b2def6b18a428ac77",
        "4e7b3e00288586ebd08712fdd0374a03",
        "b2b6027bc5c5109e529d4dc6358b12c3",
        "4f2d8ab171c80ec8364f7c12e35b23ad"
      ]
    },
    "customer_unique_id": {
      "type": "text",
      "null_count": 0,
      "null_percentage": 0.0,
      "unique_count": 96096,
      "sample_values": [
        "861eff4711a542e4b93843c6dd7febb0",
        "290c77bc529b7ac935b93aa66c333dc3",
        "060e732b5b29e8181a18229c7b0b2b5e",
        "259dac757896d24d7702b9acbbff3f3c",
        "345ecd01c38d18a9036ed96c73b8d066"
      ]
    },
    "customer_zip_code_prefix": {
      "type": "numeric",
      "null_count": 0,
      "null_percentage": 0.0,
   

---
## 5. Relationship discovery  (`src/schema/relationships.py`)

Two strategies are combined:

1. **Explicit** — foreign-key constraints already in the DB schema.
2. **Inferred** — for every column pair across tables, a score is computed from:
   - naming similarity (e.g., `order_id == order_id`, `product_id == products.product_id`)
   - value overlap (sample 100 values from A, check existence in B)
   - type match
   - uniqueness statistics from the profile

Confidence = `0.3 * naming + 0.7 * overlap`; threshold > 0.5.

Run `scripts/discover_relationships.py` to produce `data/relationships.json`, or inline:

In [8]:
from src.schema.relationships import RelationshipDiscoverer

discoverer = RelationshipDiscoverer(engine, schema, profile)
explicit = discoverer.discover_explicit()
inferred = discoverer.discover_inferred()
all_rels  = discoverer.discover_all()

print(f'Explicit FKs found:  {len(explicit)}')
print(f'Inferred  rels:      {len(inferred)}')
print(f'Total (deduplicated):{len(all_rels)}')
print()
for r in all_rels[:8]:
    print(f'  {r.source_table}.{r.source_column} -> {r.target_table}.{r.target_column}  '
          f'[{r.relationship_type}] conf={r.confidence:.3f} ({r.source})')

INFO:src.schema.relationships:Discovering explicit relationships...
INFO:src.schema.relationships:Discovering inferred relationships...
INFO:src.schema.relationships:Discovering explicit relationships...
INFO:src.schema.relationships:Discovering inferred relationships...


Explicit FKs found:  0
Inferred  rels:      23
Total (deduplicated):23

  customers.customer_id -> orders.customer_id  [one_to_one] conf=0.940 (inferred)
  leads_closed.mql_id -> leads_qualified.mql_id  [one_to_one] conf=0.940 (inferred)
  leads_closed.seller_id -> order_items.seller_id  [many_to_many] conf=0.611 (inferred)
  leads_closed.seller_id -> sellers.seller_id  [one_to_one] conf=0.611 (inferred)
  order_items.order_id -> order_payments.order_id  [many_to_many] conf=0.940 (inferred)
  order_items.order_id -> order_reviews.order_id  [many_to_many] conf=0.912 (inferred)
  order_items.order_id -> orders.order_id  [many_to_one] conf=0.940 (inferred)
  order_items.product_id -> products.product_id  [many_to_one] conf=0.940 (inferred)


---
## 6. Semantic mapping  (`src/schema/semantic_mapper.py`)

`SemanticMapper` calls the LLM once per table and once per column to generate
business-friendly descriptions (e.g., *"This table contains all customer orders placed on the Olist platform."*).

The output is `data/semantic_schema.json`, which the retriever indexes.

> **Note:** Generating this file requires a working LLM API key (set `LLM_API_KEY`).
> We skip the LLM call below and build a minimal semantic schema inline for demonstration.

In [9]:
# Build a fallback semantic schema from the raw schema + profile (no LLM needed)
def build_basic_semantic_schema(schema, profile):
    profile_map = {p['table']: p for p in profile['tables']}
    sem = {'database_type': schema.database_type, 'tables': []}
    for t in schema.tables:
        row_count = profile_map.get(t.name, {}).get('row_count', 'unknown')
        table_desc = f"Table '{t.name}' ({row_count} rows)."
        col_descs = []
        for c in t.columns:
            cprof = profile_map.get(t.name, {}).get('columns', {}).get(c.name, {})
            ctype = cprof.get('type', 'unknown')
            col_descs.append({'column': c.name, 'data_type': c.data_type,
                             'description': f"Column '{c.name}' of type {ctype}."})
        sem['tables'].append({'table': t.name, 'description': table_desc, 'columns': col_descs})
    return sem

semantic_schema = build_basic_semantic_schema(schema, profile)

# Save it so the retriever can load it
os.makedirs('data', exist_ok=True)
with open('data/semantic_schema.json', 'w') as f:
    json.dump(semantic_schema, f, indent=2)

print(f'Wrote data/semantic_schema.json ({len(semantic_schema["tables"])} tables)')
print(json.dumps(semantic_schema['tables'][0], indent=2)[:500])

Wrote data/semantic_schema.json (11 tables)
{
  "table": "customers",
  "description": "Table 'customers' (99441 rows).",
  "columns": [
    {
      "column": "customer_id",
      "data_type": "TEXT",
      "description": "Column 'customer_id' of type identifier."
    },
    {
      "column": "customer_unique_id",
      "data_type": "TEXT",
      "description": "Column 'customer_unique_id' of type text."
    },
    {
      "column": "customer_zip_code_prefix",
      "data_type": "INTEGER",
      "description": "Column 'customer_zip_code_p


---
## 7. Schema retrieval  (`src/retrieval/retriever.py`)

`SchemaRetriever` embeds all table and column descriptions using
**`sentence-transformers/all-MiniLM-L6-v2`** (384-dim), builds a **FAISS** flat-L2 index,
and returns the top-k most relevant schema documents for any natural-language question.

This is the **R** in RAG — it keeps the LLM prompt small and focused.

In [10]:
from src.retrieval.retriever import SchemaRetriever

retriever = SchemaRetriever('data/semantic_schema.json')
print(f'Indexed {len(retriever.documents)} documents')
print(f'FAISS index size: {retriever.index.ntotal}')

INFO:numexpr.utils:NumExpr defaulting to 8 threads.
INFO:datasets:TensorFlow version 2.21.0 available.
INFO:faiss.loader:Loading faiss.
INFO:faiss.loader:Successfully loaded faiss.
INFO:sentence_transformers.base.model:No device provided, using mps
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Re

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

INFO:src.retrieval.retriever:Index built with 81 documents.


Indexed 81 documents
FAISS index size: 81


In [11]:
# Try a few example queries
queries = [
    "How many products did we sell?",
    "What is the total revenue?",
    "Show me orders from São Paulo",
    "What are the most common product categories?",
]

for q in queries:
    results = retriever.retrieve(q, top_k=3)
    print(f'\nQ: {q}')
    for doc, score in results:
        print(f'  [{score:.3f}] {doc["content"][:80]}')

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q: How many products did we sell?
  [0.756] Table products: Table 'products' (32951 rows).
  [1.093] Table sellers: Table 'sellers' (3095 rows).
  [1.366] Table order_items: Table 'order_items' (112650 rows).


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q: What is the total revenue?
  [1.135] Table leads_closed, Column declared_monthly_revenue: Column 'declared_monthly_re
  [1.372] Table products: Table 'products' (32951 rows).
  [1.372] Table sellers: Table 'sellers' (3095 rows).


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q: Show me orders from São Paulo
  [1.087] Table orders: Table 'orders' (99441 rows).
  [1.313] Table order_items: Table 'order_items' (112650 rows).
  [1.345] Table orders, Column order_id: Column 'order_id' of type identifier.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Q: What are the most common product categories?
  [0.859] Table products, Column product_category_name: Column 'product_category_name' of 
  [1.001] Table products: Table 'products' (32951 rows).
  [1.095] Table product_category_name_translation, Column product_category_name_english: C


---
## 8. SQL generation  (`src/agent/sql_generator.py`)

`SQLGenerator` takes the question, retrieved schema docs, and relationship metadata,
composes a prompt, and asks the LLM to return a JSON object with:
```json
{
  "sql": "SELECT ...",
  "tables_used": [...],
  "columns_used": [...],
  "reasoning_summary": "...",
  "assumptions": "..."
}
```

Let's examine the prompt template:

In [12]:
import inspect, textwrap
from src.agent.sql_generator import SQLGenerator

src = inspect.getsource(SQLGenerator.generate)
print(textwrap.dedent(src))

def generate(self, question: str, retrieved_schema: List[Dict[str, Any]], relationships: List[RelationshipMetadata], dialect: str = "sqlite") -> Dict[str, Any]:
    """
    Generates a SQL query based on the provided context.
    """
    logger.info("Generating SQL...")

    # Format schema context
    schema_context = "\n".join([
        f"- {doc['content']}" for doc, score in retrieved_schema
    ])

    # Format relationship context
    rel_context = "\n".join([
        f"- {rel.source_table}.{rel.source_column} -> {rel.target_table}.{rel.target_column} ({rel.relationship_type})"
        for rel in relationships
    ])

    prompt = (
        f"User Question: {question}\n\n"
        f"Relevant Schema:\n{schema_context}\n\n"
        f"Table Relationships:\n{rel_context}\n\n"
        f"Database Dialect: {dialect}\n\n"
        "Generate a read-only SQL query to answer the question. "
        "Return the result in JSON format with the following keys: "
        "'sql', 'tables_used', 'co

The `generate()` method builds `schema_context` and `rel_context` strings, then calls `LLMClient.complete()`.

> Generating SQL requires a working LLM. If `LLM_API_KEY` is not set the call will fail.
> We demonstrate the prompt construction without executing the LLM call.

In [13]:
demo_question = "What is the total revenue from all orders?"
retrieved = retriever.retrieve(demo_question, top_k=5)
retrieved_simplified = [(doc, score) for doc, score in retrieved]

# Show what the LLM would see
schema_context = '\n'.join([f"- {doc['content']}" for doc, score in retrieved_simplified])
rel_context = '\n'.join([
    f"- {r.source_table}.{r.source_column} -> {r.target_table}.{r.target_column} ({r.relationship_type})"
    for r in all_rels[:20]
])

print('=== Schema context ===')
print(schema_context)
print('\n=== Relationship context (first 20) ===')
print(rel_context[:600])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

=== Schema context ===
- Table orders: Table 'orders' (99441 rows).
- Table order_items: Table 'order_items' (112650 rows).
- Table sellers: Table 'sellers' (3095 rows).
- Table order_payments: Table 'order_payments' (103886 rows).
- Table leads_closed, Column declared_monthly_revenue: Column 'declared_monthly_revenue' of type numeric.

=== Relationship context (first 20) ===
- customers.customer_id -> orders.customer_id (one_to_one)
- leads_closed.mql_id -> leads_qualified.mql_id (one_to_one)
- leads_closed.seller_id -> order_items.seller_id (many_to_many)
- leads_closed.seller_id -> sellers.seller_id (one_to_one)
- order_items.order_id -> order_payments.order_id (many_to_many)
- order_items.order_id -> order_reviews.order_id (many_to_many)
- order_items.order_id -> orders.order_id (many_to_one)
- order_items.product_id -> products.product_id (many_to_one)
- order_items.seller_id -> sellers.seller_id (many_to_one)
- order_payments.order_id -> order_items.order_id 


---
## 9. SQL validation  (`src/agent/validator.py`)

`SQLValidator` runs three checks:
1. **Syntax** — parsed with `sqlglot` (AST must build without error).
2. **Read-only** — root AST node must be `SELECT`.
3. **Banned keywords** — rejects queries containing DROP/DELETE/UPDATE/INSERT/ALTER/TRUNCATE/CREATE/GRANT/REVOKE.

In [14]:
from src.agent.validator import SQLValidator

validator = SQLValidator()

test_cases = [
    ("SELECT COUNT(*) FROM orders",                     "Should pass"),
    ("SELECT * FROM orders WHERE order_status = 'delivered'", "Should pass"),
    ("INSERT INTO orders VALUES (1, 'test')",           "Should fail (INSERT)"),
    ("DROP TABLE orders",                               "Should fail (DROP)"),
    ("SELCT * FROM orders",                             "Should fail (syntax)"),
]

for sql, desc in test_cases:
    ok, err = validator.validate(sql)
    status = 'PASS' if ok else f'FAIL: {err}'
    print(f'  {desc:30s}  ->  {status}')

  Should pass                     ->  PASS
  Should pass                     ->  PASS
  Should fail (INSERT)            ->  FAIL: Only SELECT queries are allowed.
  Should fail (DROP)              ->  FAIL: Only SELECT queries are allowed.
  Should fail (syntax)            ->  FAIL: SQL Syntax Error: Required keyword: 'expression' missing for <class 'sqlglot.expressions.core.Mul'>. Line 1, Col: 12.
  SELCT * FROM orders


---
## 10. SQL repair  (`src/agent/repair.py`)

When validation or execution fails, `SQLRepairer.repair()` sends the broken SQL + error message
back to the LLM and asks it to fix the query. It retries up to 3 times,
re-validating each repair attempt.

In [15]:
import inspect, textwrap
from src.agent.repair import SQLRepairer

print(textwrap.dedent(inspect.getsource(SQLRepairer.repair)))

def repair(self, original_sql: str, error: str, question: str, schema_context: str, max_attempts: int = 3) -> Optional[str]:
    """
    Attempts to repair a failing SQL query.
    """
    current_sql = original_sql

    for attempt in range(1, max_attempts + 1):
        logger.info(f"Repair attempt {attempt}/{max_attempts}...")

        prompt = (
            f"Original SQL: {current_sql}\n"
            f"Database Error: {error}\n"
            f"User Question: {question}\n"
            f"Schema Context: {schema_context}\n\n"
            "Please repair the SQL query to fix the error. Return only the corrected SQL query."
        )
        system_prompt = "You are an expert SQL debugger. Fix the provided SQL based on the error message."

        current_sql = self.llm_client.complete(prompt, system_prompt).strip()

        # Clean up LLM output (remove markdown code blocks)
        if current_sql.startswith("```sql"):
            current_sql = current_sql.split("\n", 1)[1].rsplit("\n", 

Note the markdown-fence stripping logic — LLMs often wrap SQL in ```sql fences, which
would break execution. The repairer cleans that up.

---
## 11. Result analysis  (`src/agent/analyst.py`)

`ResultAnalyst` takes the original question, the executed SQL, and the raw row data,
and asks the LLM to produce a concise natural-language summary.

In [16]:
import inspect, textwrap
from src.agent.analyst import ResultAnalyst

print(textwrap.dedent(inspect.getsource(ResultAnalyst.analyze)))

def analyze(self, question: str, sql: str, results: List[Dict[str, Any]]) -> str:
    """
    Converts the SQL result set into a natural language answer.
    """
    logger.info("Analyzing results...")

    prompt = (
        f"User Question: {question}\n"
        f"SQL Query: {sql}\n"
        f"Query Results: {results}\n\n"
        "Provide a natural language answer to the user question based on these results. "
        "Be precise, include numerical values where applicable, and mention any caveats."
    )
    system_prompt = "You are a data analyst. Provide clear, accurate, and professional answers based on the provided data."

    return self.llm_client.complete(prompt, system_prompt).strip()



Example of what a raw result looks like before analysis:

In [17]:
with engine.connect() as conn:
    rows = [dict(r._mapping) for r in conn.execute(text(
        "SELECT order_status, COUNT(*) AS order_count FROM orders GROUP BY order_status ORDER BY order_count DESC"
    ))]

import pandas as pd
pd.DataFrame(rows)

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


---
## 12. Visualization  (`src/analytics/visualization.py`)

`Visualizer` asks the LLM whether a chart would be helpful, and if so returns
a Plotly-compatible spec dict: `{type, x, y, title, labels}`.
If no chart is warranted, it returns `"NONE"`.

In [18]:
import inspect, textwrap
from src.analytics.visualization import Visualizer

print(textwrap.dedent(inspect.getsource(Visualizer.generate_chart_spec)))

def generate_chart_spec(self, question: str, results: List[Dict[str, Any]]) -> Optional[Dict[str, Any]]:
    """
    Generates a Plotly-compatible chart specification if useful.
    """
    logger.info("Determining if visualization is useful...")

    prompt = (
        f"User Question: {question}\n"
        f"Data Results: {results}\n\n"
        "If this data can be visualized effectively (e.g., time series, distribution, comparison), "
        "return a JSON chart specification with 'type' (line, bar, pie, scatter), 'x', 'y', 'title', and 'labels'. "
        "If no visualization is useful, return 'NONE'."
    )
    system_prompt = "You are a data visualization expert. Return a valid JSON chart spec or 'NONE'."

    response = self.llm_client.complete(prompt, system_prompt).strip()

    if response == "NONE":
        return None

    try:
        import json
        return json.loads(response)
    except json.JSONDecodeError:
        logger.error(f"Failed to parse chart spec JSON: {re

---
## 13. The orchestrator  (`src/agent/orchestrator.py`)

`AIAnalystAgent.ask()` ties everything together into this pipeline:

```
Question
  |  (if follow-up: rewrite to standalone via LLM)
  v
Retrieve relevant schema docs (FAISS)
  v
Generate SQL (LLM + schema + relationships)
  v
Validate SQL (sqlglot + read-only + banned keywords)
  |  if invalid -> Repair (LLM, up to 3 attempts)
  v
Execute SQL
  |  if error -> Repair (LLM) and retry
  v
Analyze results (LLM -> natural language)
  v
Optionally generate chart spec (LLM)
  v
Return {question, sql, answer, data, chart}
```

It also maintains `self.history` for multi-turn conversations.

In [19]:
import inspect, textwrap
from src.agent.orchestrator import AIAnalystAgent

src = inspect.getsource(AIAnalystAgent.ask)
print(textwrap.dedent(src))

def ask(self, question: str) -> Dict[str, Any]:
    """
    The full pipeline from question to answer.
    """
    # Handle multi-turn by rewriting the question based on history
    final_question = question
    if self.history:
        final_question = self._rewrite_question(question)

    logger.info(f"Processing question: {final_question}")

    # 1. Retrieve relevant schema
    retrieved_schema = self.retriever.retrieve(final_question)

    # 2. Generate SQL
    gen_result = self.generator.generate(final_question, retrieved_schema, self.relationships)
    sql = gen_result.get("sql", "")

    if not sql:
        return {"error": "Could not generate SQL query.", "details": gen_result}

    # 3. Validate SQL
    is_valid, error_msg = self.validator.validate(sql)
    if not is_valid:
        logger.warning(f"SQL validation failed: {error_msg}. Attempting repair...")
        schema_ctx = "\n".join([doc['content'] for doc, score in retrieved_schema])
        sql = self.repairer.repair(sq

Show the question-rewriting logic for follow-ups:

In [20]:
print(textwrap.dedent(inspect.getsource(AIAnalystAgent._rewrite_question)))

def _rewrite_question(self, question: str) -> str:
    """
    Rewrites a follow-up question into a standalone question using history.
    """
    history_str = "\n".join([f"Q: {h['question']}\nA: {h['answer']}" for h in self.history[-3:]])
    prompt = (
        f"Conversation History:\n{history_str}\n\n"
        f"Follow-up Question: {question}\n\n"
        "Rewrite the follow-up question into a standalone, complete question that can be answered without the history."
    )
    system_prompt = "You are a query re-writer. Output only the standalone question."
    return self.llm_client.complete(prompt, system_prompt).strip()



---
## 14. Full agent demo  (requires `LLM_API_KEY`)

If you have a valid API key set as the `LLM_API_KEY` environment variable,
you can instantiate the full agent and ask questions:

In [21]:
has_key = bool(os.environ.get('LLM_API_KEY', '').strip()) and os.environ.get('LLM_API_KEY') != 'your-api-key-here'
print(f'LLM API key configured: {has_key}')

if has_key:
    agent = AIAnalystAgent(
        database_url='sqlite:///data/olist.db',
        semantic_schema_path='data/semantic_schema.json',
    )
    result = agent.ask("How many orders have been delivered?")
    print(json.dumps(result, indent=2, default=str))
else:
    print('Set LLM_API_KEY to run this cell.')

LLM API key configured: False
Set LLM_API_KEY to run this cell.


---
## 15. LLM client & config  (`src/agent/llm_client.py`, `src/config.py`)

The client is a thin wrapper around the `openai` Python SDK, configured
to work with any OpenAI-compatible endpoint (OpenAI, Azure, Ollama, etc.).

| Env var | Default | Purpose |
|---|---|---|
| `LLM_API_KEY` | `your-api-key-here` | API key |
| `LLM_BASE_URL` | `https://api.openai.com/v1` | Endpoint URL |
| `LLM_MODEL` | `gpt-4o` | Model name |

In [22]:
from src.config import load_config
from src.agent.llm_client import LLMClient

config = load_config()
print(f'base_url: {config.base_url}')
print(f'model:    {config.model}')
print(f'temp:     {config.temperature}')
print(f'max_tok:  {config.max_tokens}')

base_url: https://api.openai.com/v1
model:    gpt-4o
temp:     0.0
max_tok:  1000


---
## 16. Evaluation framework  (`src/evaluation/`)

### Benchmark set (`src/evaluation/benchmark.py`)

A hardcoded set of 5 `EvalCase` objects covering easy/hard/impossible questions.

In [23]:
from src.evaluation.benchmark import get_olist_eval_set

eval_set = get_olist_eval_set()
for case in eval_set:
    print(f'[{case.difficulty:6s}] [{case.category:11s}] {case.question}')

[easy  ] [simple     ] How many products are in the database?
[easy  ] [aggregation] What is the total revenue from all orders?
[medium] [join       ] Which product category generated the most revenue?
[medium] [temporal   ] How many orders were placed in 2017?
[easy  ] [impossible ] What is the average age of the customers?


### Metrics (`src/evaluation/metrics.py`)

In [24]:
from src.evaluation.metrics import EvalMetrics

# Execution accuracy: direct equality of result lists
print('Execution accuracy (equal):  ', EvalMetrics.calculate_execution_accuracy([{'a':1}], [{'a':1}]))
print('Execution accuracy (differ): ', EvalMetrics.calculate_execution_accuracy([{'a':1}], [{'a':2}]))

# Retrieval recall / precision
retrieved = ['orders.order_id', 'orders.order_status', 'orders.customer_id']
required  = ['orders.order_id', 'orders.customer_id']
print(f'Recall@3:  {EvalMetrics.calculate_retrieval_recall(retrieved, required):.2f}')
print(f'Precision@3: {EvalMetrics.calculate_retrieval_precision(retrieved, required):.2f}')

Execution accuracy (equal):   True
Execution accuracy (differ):  False
Recall@3:  1.00
Precision@3: 0.67


### Runner (`src/evaluation/runner.py`)

Iterates over the eval set, calls `agent.ask()` for each question, measures execution accuracy and latency.
Semantic accuracy uses an LLM judge (or falls back to exact match).

> Requires a working agent (`LLM_API_KEY`).

In [25]:
import inspect, textwrap
from src.evaluation.runner import EvalRunner
print(textwrap.dedent(inspect.getsource(EvalRunner.run)))

def run(self, eval_set: List[Any]) -> Dict[str, Any]:
    logger.info(f"Running evaluation on {len(eval_set)} cases...")

    results = []
    summary = {
        "execution_accuracy": 0,
        "retrieval_recall": 0,
        "semantic_accuracy": 0,
        "total_cases": len(eval_set)
    }

    for case in eval_set:
        logger.info(f"Evaluating: {case.question}")

        start_time = time.time()
        agent_res = self.agent.ask(case.question)
        latency = time.time() - start_time

        # 1. Execute Gold SQL for ground truth
        gold_res = None
        if case.gold_sql != "SELECT 'NOT_POSSIBLE'":
            with self.agent.engine.connect() as conn:
                res = conn.execute(text(case.gold_sql))
                gold_res = [dict(row._mapping) for row in res]

        # 2. Calculate Metrics
        # Execution Accuracy
        exec_acc = self.metrics.calculate_execution_accuracy(
            agent_res.get("data"), gold_res
        ) if "error" not in agent_r

### Experiment baselines (`src/evaluation/experiments.py`)

Compares three configurations:
1. **No schema** — question goes straight to LLM with no context.
2. **Full schema** — all tables/columns are injected into the prompt.
3. **Full system (RAG)** — only the top-k relevant docs are retrieved.

In [26]:
import inspect, textwrap
from src.evaluation.experiments import ExperimentRunner
print(textwrap.dedent(inspect.getsource(ExperimentRunner.run_all_experiments)))

def run_all_experiments(self):
    eval_set = get_olist_eval_set()

    b1 = self.run_baseline_1(eval_set)
    b2 = self.run_baseline_2(eval_set)
    full = self.run_system_full(eval_set)

    print("\n=== Experiment Results ===")
    print(f"Baseline 1 (No Schema): {b1:.2%}")
    print(f"Baseline 2 (Full Schema): {b2:.2%}")
    print(f"Full System (RAG):       {full:.2%}")
    print("==========================\n")



### Robustness tests (`src/evaluation/robustness.py`)

`RobustnessTester` probes the agent with ambiguous questions ("How are things doing?"),
impossible questions ("What is the average age of customers?" — no age column),
and schema drift simulations.

In [27]:
from src.evaluation.robustness import RobustnessTester
print(textwrap.dedent(inspect.getsource(RobustnessTester)))

class RobustnessTester:
    """
    Tests the system against adversarial or challenging scenarios.
    """
    def __init__(self, agent: AIAnalystAgent):
        self.agent = agent

    def test_ambiguous_question(self, question: str):
        logger.info(f"Testing ambiguous question: {question}")
        return self.agent.ask(question)

    def test_impossible_question(self, question: str):
        logger.info(f"Testing impossible question: {question}")
        return self.agent.ask(question)

    def test_schema_drift(self, table_name: str, col_name: str, new_name: str):
        """
        Simulates a schema change by mocking the retriever's returned content.
        """
        logger.info(f"Testing schema drift: {col_name} -> {new_name}")
        # In a real system, we'd modify the DB or the semantic schema
        # Here we can just test if the agent handles a mismatched column name in the result
        pass



---
## 17. Apps

### FastAPI (`app/main.py`)

Three endpoints:
- `GET /health` — returns `{"status": "healthy"}`
- `POST /ask` — body `{"question": "..."}` → agent result
- `GET /schema` — dumps `data/semantic_schema.json`

### Streamlit (`app/streamlit_app.py`)

A chat-based UI with:
- Sidebar: DB URL / schema path / reset button
- Chat history with expandable SQL + dataframe + chart views
- Session-scoped `AIAnalystAgent` in `st.session_state`

> **Bug note:** `app/main.py` line 23 references `Dict[str, Any]` in the `AgentResponse` model
> but `Dict` is not imported from `typing`. This would raise a `NameError` at startup.

---
## 18. Architecture summary

```
User question
     │
     ▼
┌────────────────────────────────────────────┐
│            AIAnalystAgent.ask()            │
│  (src/agent/orchestrator.py)               │
│                                            │
│  1. Rewrite follow-up → standalone (LLM)   │
│  2. Retrieve schema docs (FAISS/RAG)       │
│     └─ src/retrieval/retriever.py          │
│  3. Generate SQL (LLM + schema + rels)     │
│     └─ src/agent/sql_generator.py          │
│  4. Validate SQL (sqlglot)                 │
│     └─ src/agent/validator.py              │
│  5. Repair if invalid (LLM, max 3)         │
│     └─ src/agent/repair.py                 │
│  6. Execute SQL (SQLAlchemy)                │
│  7. Analyze results → NL answer (LLM)       │
│     └─ src/agent/analyst.py                │
│  8. Generate chart spec (LLM)              │
│     └─ src/analytics/visualization.py      │
│                                            │
│  Returns: {question, sql, answer, data,     │
│            chart}                          │
└────────────────────────────────────────────┘

Supporting layers:
  src/schema/*         — extraction, profiling, relationships, semantic mapping
  src/config.py        — LLM config (env vars)
  src/analytics/analysis.py — MetricLayer (YAML-based business metrics)
  scripts/*            — one-shot setup utilities
  src/evaluation/*     — benchmark, metrics, runner, experiments, robustness
  app/*                — FastAPI + Streamlit front-ends
```

### Key design decisions
- **RAG over full schema dump** — keeps LLM context small; validated by experiment baselines.
- **sqlglot for validation** — AST-based, supports multiple dialects; better than regex.
- **Repair loop** — self-healing via LLM error feedback; 3-attempt limit.
- **Semantic schema via LLM** — business descriptions improve retrieval quality.
- **Relationship inference** — value-overlap sampling catches implicit FKs the DB misses.

### Known gaps
- `requirements.txt` and `pyproject.toml` are empty.
- `data/semantic_schema.json` is not checked into the repo (must be generated).
- `app/main.py` has a `NameError` bug (`Dict` not imported).
- Retrieval recall is not measured in the eval runner (hardcoded to 0.0).
- The `MetricLayer` in `analysis.py` is a stub (no metrics YAML in repo).